In [1]:
import psutil
import tracemalloc
import numpy as np
from ase.db import connect
import os

In [2]:
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1e9:.1f} GB")
print(f"Available RAM: {ram.available / 1e9:.1f} GB")

Total RAM: 17.2 GB
Available RAM: 5.0 GB


In [3]:
data_dir = os.path.expanduser("~/Desktop/data/train_4m")
shards = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".aselmdb")])
db = connect(shards[0])

In [7]:
size_buckets = {
    (0, 10): [],
    (11, 20): [],
    (21, 30): [],
    (31, 50): [],
    (51, 75): [],
    (76, 100): [],
    (101, 150): [],
    (151, 200): [],
    (201, 350): [],
}

for row in db.select():
    n = row.data["num_atoms"]
    for (low, high), samples in size_buckets.items():
        if low <= n <= high and len(samples) < 3:
            samples.append(row)
    if all(len(v) >= 15 for v in size_buckets.values()):
        break

def measure_molecule_memory(row):
    tracemalloc.start()
    
    atoms = row.toatoms()
    data = {
        "positions": atoms.positions.copy(),
        "atomic_numbers": atoms.get_atomic_numbers().copy(),
        "forces": row.forces.copy(),
        "energy": row.energy,
        "charge": row.data["charge"],
        "spin": row.data["spin"],
    }
    
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1e3, data  # in KB

print(f"{'Atom range':<15} {'Memory/mol (KB)':<20} {'Est. 100k mols (GB)':<25} {'Est. 1M mols (GB)'}")
print("-" * 75)

for (low, high), samples in size_buckets.items():
    if not samples:
        print(f"{f'{low}-{high}':<15} no samples found")
        continue
    mem_kb = np.mean([measure_molecule_memory(s)[0] for s in samples])
    est_100k = (mem_kb * 100_000) / 1e6
    est_1m = (mem_kb * 1_000_000) / 1e6
    print(f"{f'{low}-{high}':<15} {mem_kb:<20.1f} {est_100k:<25.2f} {est_1m:.2f}")

Atom range      Memory/mol (KB)      Est. 100k mols (GB)       Est. 1M mols (GB)
---------------------------------------------------------------------------
0-10            4.9                  0.49                      4.90
11-20           5.4                  0.54                      5.36
21-30           6.8                  0.68                      6.81
31-50           9.8                  0.98                      9.79
51-75           12.7                 1.27                      12.67
76-100          16.4                 1.64                      16.36
101-150         20.5                 2.05                      20.49
151-200         27.3                 2.73                      27.26
201-350         42.8                 4.28                      42.76
